this is going to be phase one
1. Understand dataset
2. Clean data
3. Preprocess text
4. Split dataset
5. Convert text → vectors (TF-IDF)

In [106]:
# let load the librarys in this code book

# ! pip install kaggle

import kagglehub


In [107]:
# commands set up your Kaggle API credentials securely so you can use Kaggle datasets and competitions directly from Colab or your notebook
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json


In [108]:
#✅ Verify the setup
# !kaggle datasets list


In [109]:
#API to fetch the data from kaggle (downlode the zip file of the dataset)
!kaggle datasets download -d kazanova/sentiment140


Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
sentiment140.zip: Skipping, found more recently modified local copy (use --force to force download)


In [110]:
# #extrating the compress dataset (unzip the file )
# from zipfile import ZipFile   #This imports Python’s built‑in zipfile module, which lets you work with .zip archives (open them, extract files, list contents, etc.
# dataset ='/content/sentiment140.zip'

# # Open the zip file
# with ZipFile(dataset, 'r') as zip_ref:
#   # Extract all contents into a folder
#   zip_ref.extractall("sentiment140")
#   print('The dataset is extracted')

In [111]:
import pandas as pd
path = kagglehub.dataset_download("kazanova/sentiment140")
df = pd.read_csv(path + "/training.1600000.processed.noemoticon.csv",
                 encoding="latin-1", header=None)

df.columns = ["target", "id", "date", "flag", "user", "text"]
# Quick look at the data
print(df.head())


Using Colab cache for faster access to the 'sentiment140' dataset.
   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [112]:
# Count how many positive vs negative tweets
print(df['target'].value_counts())

target
0    800000
4    800000
Name: count, dtype: int64


In [113]:
print(df.head(10)) # to check starting 10 data

# Count how many positive vs negative tweets
print(df['target'].value_counts())

# Look at only the text column
print(df['text'].head(10))

# Filter negative tweets
neg_tweets = df[df['target'] == 0]
print(neg_tweets.head(5))


   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
5       0  1467811372  Mon Apr 06 22:20:00 PDT 2009  NO_QUERY   
6       0  1467811592  Mon Apr 06 22:20:03 PDT 2009  NO_QUERY   
7       0  1467811594  Mon Apr 06 22:20:03 PDT 2009  NO_QUERY   
8       0  1467811795  Mon Apr 06 22:20:05 PDT 2009  NO_QUERY   
9       0  1467812025  Mon Apr 06 22:20:09 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man..

In [114]:
# All important library that we need later on
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from nltk.stem import WordNetLemmatizer

from sklearn.metrics import classification_report

from sklearn.metrics import confusion_matrix



In [115]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

# print the stopWatch in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


Data processing


In [116]:
#loading the data from csv file to pandas
twitter_data = pd.read_csv(path + '/training.1600000.processed.noemoticon.csv', encoding='iso-8859-1')

# checking the number of rows and columns
twitter_data.shape


(1599999, 6)

In [117]:
# printing the frist 5 rows of the data
twitter_data.head(5)

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [118]:
# naming the column name
column_names = ["target", "id", "date", "flag", "user", "text"]
twitter_data = pd.read_csv(path + '/training.1600000.processed.noemoticon.csv', names = column_names , encoding='iso-8859-1', header=None)
# twitter_data.column = column_names

print(twitter_data.head())

# checking the number of rows and columns
twitter_data.shape

   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


(1600000, 6)

In [119]:
# checking wether there is any null or missing values
twitter_data.isnull().sum()

,0
target,0
id,0
date,0
flag,0
user,0
text,0


we don't have any missing values but if we have
id best to remove the row or fill the missing values
##This remove the row
  twitter_data = twitter_data.dropna()

  # Fill with empty string in text column
twitter_data['text'] = twitter_data['text'].fillna("")

# Fill numeric columns with 0
twitter_data['target'] = twitter_data['target'].fillna(0)


In [120]:

# checking the distribution of target
twitter_data['target'].value_counts()


,count
target,
0,800000
4,800000


# convert target "4" to "1" so they seem 0 and 1  

In [121]:
# twitter_data['target'] = twitter_data['target'].replace(4,1)
# twitter_data['target'].value_counts()

twitter_data.replace({'target':{4:1}}, inplace=True)
twitter_data['target'].value_counts()

,count
target,
0,800000
1,800000


0 --> NEGETIVE TWEET

1 --> POSITIVE TWEET

# Stemming
  - it us a process of reducing a world to it's Root word

    - eg:- actor, actress, acting = act   



In [122]:
# from nltk.stem.porter import PorterStemmer
stop_words = set(stopwords.words('english'))

negation_words = {'not', 'no', 'nor', 'never'}

stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()


In [123]:
# import re


def clean_tweet(content):

    # lowercase
    content = content.lower()

    # remove urls
    content = re.sub(r'http\S+|www\S+', '', content)

    # remove mentions
    content = re.sub(r'@\w+', '', content)

    # keep hashtag words
    content = re.sub(r'#', '', content)

    # remove numbers
    content = re.sub(r'\d+', '', content)

    # remove punctuation
    content = re.sub(r'[^a-zA-Z\s]', ' ', content)

    # remove extra spaces
    content = re.sub(r'\s+', ' ', content).strip()

    words = content.split()

    cleaned_words = []

    for word in words:

        if word not in stop_words:

            word = lemmatizer.lemmatize(word)

            cleaned_words.append(word)

    return " ".join(cleaned_words)

In [124]:
twitter_data['cleaned_content'] = twitter_data['text'].apply(clean_tweet) #take loong time

In [125]:
twitter_data.head(10)

,target,id,date,flag,user,text,cleaned_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",bummer shoulda got david carr third day
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset update facebook texting might cry result...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,dived many time ball managed save rest go bound
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",no not behaving mad see
5,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew,not whole crew
6,0,1467811592,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,mybirch,Need a hug,need hug
7,0,1467811594,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,coZZ,@LOLTrish hey long time no see! Yes.. Rains a...,hey long time no see yes rain bit bit lol fine...
8,0,1467811795,Mon Apr 06 22:20:05 PDT 2009,NO_QUERY,2Hood4Hollywood,@Tatiana_K nope they didn't have it,nope
9,0,1467812025,Mon Apr 06 22:20:09 PDT 2009,NO_QUERY,mimismo,@twittera que me muera ?,que muera


In [128]:
print(twitter_data[['target', 'cleaned_content']].head())


   target                                    cleaned_content
0       0            bummer shoulda got david carr third day
1       0  upset update facebook texting might cry result...
2       0    dived many time ball managed save rest go bound
3       0                    whole body feel itchy like fire
4       0                            no not behaving mad see


# Separating the data and the lable

In [129]:
x = twitter_data['cleaned_content'].values
y=twitter_data['target'].values

In [130]:
print(x,y)

['bummer shoulda got david carr third day'
 'upset update facebook texting might cry result school today also blah'
 'dived many time ball managed save rest go bound' ...
 'ready mojo makeover ask detail'
 'happy th birthday boo alll time tupac amaru shakur'
 'happy charitytuesday'] [0 0 0 ... 1 1 1]


# Spliting data into traning and testing data

In [131]:
#from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2, stratify=y, random_state=2)

In [132]:
print(x.shape, X_train.shape, X_test.shape)
print(y.shape, Y_train.shape, Y_test.shape)
print(X_train)
print(X_test)

(1600000,) (1280000,) (320000,)
(1600000,) (1280000,) (320000,)
['watch saw iv drink lil wine' ''
 'even though favourite drink think vodka coke wipe mind time think im gonna find new drink'
 ... 'eager monday afternoon'
 'hope everyone mother great day wait hear guy store tomorrow'
 'love waking folgers bad voice deeper']
['fine much time chat twitter hubby back summer amp tends dominate free time'
 'ahs may show w ruth kim amp geoffrey sanhueza'
 'maybe bay area thang dammit' ...
 'nevertheless hooray member wonderful safe trip' 'not feeling well'
 'thank']


# now as we discuss the ml modal understandinly numarical data so converting the textual data into numarical data

In [133]:
vectorizer = TfidfVectorizer(

    max_features=100000,

    ngram_range=(1,2),

    min_df=5,

    max_df=0.95,

    sublinear_tf=True
)
# vectorizer.fit(X_train)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)


In [134]:
print(X_train)
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11426161 stored elements and shape (1280000, 100000)>
  Coords	Values
  (0, 92920)	0.2628353026190577
  (0, 71696)	0.2843495343710911
  (0, 41719)	0.42180042733631223
  (0, 20025)	0.31916464586482085
  (0, 46609)	0.3347289437784056
  (0, 95852)	0.3560399057505665
  (0, 93035)	0.5757372612912077
  (2, 20025)	0.29322836273286124
  (2, 21954)	0.13741714579681163
  (2, 84297)	0.1315377795007753
  (2, 23925)	0.20466304633666305
  (2, 83690)	0.19758618822412222
  (2, 91453)	0.23176332464294194
  (2, 13754)	0.22028968517168435
  (2, 95888)	0.2577995881526525
  (2, 52522)	0.17041105272691093
  (2, 85040)	0.10688319713139975
  (2, 40368)	0.11474577117111255
  (2, 31762)	0.13219963018286712
  (2, 24834)	0.14493388917458863
  (2, 56588)	0.11804963625025036
  (2, 22100)	0.19650016835109024
  (2, 20055)	0.31602567755455446
  (2, 85528)	0.2504278219347599
  (2, 83870)	0.20984606337433812
  :	:
  (1279997, 20485)	0.5872762257438515
  (1279

Now comes the training of madal

  - we are training the logistic regeression modal which is classification modal

In [135]:
model = LogisticRegression(

    C=2,

    max_iter=2000,

    class_weight='balanced',

    n_jobs=-1
)

In [136]:
model.fit(X_train,Y_train)

LogisticRegression(C=2, class_weight='balanced', max_iter=2000, n_jobs=-1)

Modal Evalution
  - Accuracy score  

In [137]:
#Accuracy score on training data :
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [138]:
print('Accuracy score of the training data : ', training_data_accuracy)


Accuracy score of the training data :  0.8162609375


In [140]:
#Accuracy score on testing data :
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)
print(classification_report(Y_test, X_test_prediction))

print(confusion_matrix(Y_test, X_test_prediction))


              precision    recall  f1-score   support

           0       0.81      0.78      0.80    160000
           1       0.79      0.82      0.80    160000

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000

[[125358  34642]
 [ 29447 130553]]


In [141]:
print('Accuracy score of the testing data : ', test_data_accuracy)

Accuracy score of the training data :  0.799721875


#Now  we create pipline that take input and process and give output


In [147]:
tweet = input("Enter a tweet: ")

# Step 1 - Clean tweet
cleaned_tweet = clean_tweet(tweet)

# Step 2 - Convert text to TF-IDF vector
tweet_vector = vectorizer.transform([cleaned_tweet])

# Step 3 - Predict sentiment
prediction = model.predict(tweet_vector)

# Step 4 - Show result
if prediction[0] == 1:
    print("Positive Tweet 😊")
else:
    print("Negative Tweet 😠")

Enter a tweet: okey okey
Positive Tweet 😊


In [148]:
import pickle

# save trained model
pickle.dump(model, open('sentiment_model.pkl', 'wb'))

# save tfidf vectorizer
pickle.dump(vectorizer, open('tfidf_vectorizer.pkl', 'wb'))

print("Model and Vectorizer saved successfully")

Model and Vectorizer saved successfully
